In [ ]:
import pandas as pd

# Load MaAsLin3 results
df = pd.read_csv('/path/to/MaAsLin3_results/Metabolomics_ordinal_MaAsLin3/all_results_with_names.tsv', sep='\t')

# Filter significant features (q-value < 0.05) and metadata type
df = df[df['qval_individual'] < 0.05]
df_linear = df[df['metadata'] == 'Group_ord'].reset_index(drop=True)

# Select relevant columns for plotting
df_for_plot = df_linear.copy()
df_for_plot = df_for_plot[['feature_names', 'name', 'coef', 'qval_individual', 'model']]

# Subset by linear and quadratic terms
df_for_plot_L = df_for_plot[df_for_plot['name'] == 'Group_ord.L']
df_for_plot_Q = df_for_plot[df_for_plot['name'] == 'Group_ord.Q']

# Export results
df_for_plot_L.to_csv('/path/to/output/Ordinary_linear_result_for_plot.tsv', sep='\t', index=False)
df_for_plot_Q.to_csv('/path/to/output/Ordinary_quad_result_for_plot.tsv', sep='\t', index=False)

# Calc Scores

In [ ]:
import pandas as pd
import numpy as np

# 1. Load transformed data
df_path = '/path/to/MaAsLin3_results/Metabolomics_ordinal_MaAsLin3/features/data_transformed.tsv'
df = pd.read_csv(df_path, sep='\t')

# 2. Load metadata and merge group information for missing value imputation
metadata_path = '/path/to/input/metadata.tsv'
metadata = pd.read_csv(metadata_path, sep='\t')[['SampleID', 'Group']]

# 3. Feature-wise Minimum Imputation
df = df.rename(columns={'feature': 'SampleID'})
df_merged = pd.merge(df, metadata, on='SampleID', how='inner')

print(f"--- Minimum Imputation Start (Total Samples: {len(df_merged)}) ---")

# Extract numerical columns (metabolites) only
metab_cols_raw = [c for c in df_merged.columns if c not in ['SampleID', 'Group']]

# Check missing values before imputation
pre_missing = df_merged[metab_cols_raw].isnull().sum().sum()
print(f"Before imputation: {pre_missing} missing values")

# Impute missing values with the minimum value of each feature
df_cleaned = df_merged.copy()
df_cleaned[metab_cols_raw] = df_cleaned[metab_cols_raw].fillna(df_merged[metab_cols_raw].min())

# Secondary defense: fallback to global minimum if entire columns are NaN
if df_cleaned[metab_cols_raw].isnull().sum().sum() > 0:
    print("Warning: Some columns are entirely NaN. Filling with global minimum.")
    global_min = df_merged[metab_cols_raw].min().min()
    df_cleaned[metab_cols_raw] = df_cleaned[metab_cols_raw].fillna(global_min)

print(f"After imputation: {df_cleaned[metab_cols_raw].isnull().sum().sum()} missing values")

# 4. Apply feature name mapping
map_df = pd.read_csv('/path/to/MaAsLin3_results/Metabolomics_feature_name_map.tsv', sep='\t')
feature_map = dict(zip(map_df['feature'].str.strip(), map_df['feature_names'].str.strip()))

# Extract metabolite columns and apply renaming
metab_cols = [c for c in df_cleaned.columns if c not in ['SampleID', 'Group']]
df_final = df_cleaned.rename(columns={c: feature_map.get(c, c) for c in metab_cols})
df_final = df_final.set_index('SampleID')

# 5. Load MaAsLin3 coefficients
ord_res = pd.read_csv('/path/to/output/Ordinary_linear_result_for_plot.tsv', sep='\t')
ord_res = ord_res.set_index('feature_names')
weights = ord_res['coef']

# Extract intersecting features used in the analysis
common_features = weights.index.intersection(df_final.columns)
print(f"Calculating score with {len(common_features)} metabolites...")

# Calculate Weighted Sum (Dot Product)
# (Raw Transformed Abundance) * (Coefficient)
raw_score_ws = df_final[common_features].dot(weights.loc[common_features])

# 6. Organize results and apply Z-score normalization to final scores
score_df_res = raw_score_ws.reset_index()
score_df_res.columns = ['SampleID', 'Metabo_score_WS']

# Merge with final metadata
score_df_final = pd.merge(metadata, score_df_res, on='SampleID', how='inner')

# Calculate Z-score for the final score
s_mean = score_df_final['Metabo_score_WS'].mean()
s_std = score_df_final['Metabo_score_WS'].std()
score_df_final['Metabo_score_WS_z'] = (score_df_final['Metabo_score_WS'] - s_mean) / s_std

final_col_names = ['SampleID', 'Group','MetRS', 'MetRS_z']
score_df_final.columns = final_col_names
display(score_df_final.head())

# 7. Export results
output_path = '/path/to/output/Metabo_score_weighted_sum.tsv'
score_df_final.to_csv(output_path, sep='\t', index=False)
print(f"Successfully saved final Z-scored MetRS results to {output_path}")

--- Minimum Imputation Start (Total Samples: 169) ---
Before imputation: 151 missing values
After imputation: 0 missing values
Calculating score with 36 metabolites...


,SampleID,Group,MetRS,MetRS_z
0,M001,SA,-5.897131,-0.916790
1,M002,ACS,0.813109,0.259489
2,M003,SA,-2.591938,-0.337403
3,M004,HFrEF,6.432190,1.244493
4,M005,SA,-7.811895,-1.252441


Successfully saved Final Z-scored Weighted Sum scores to scores/Metabo_score_weighted_sum.tsv
